# Train DIC model

Change `MODEL_NAME` below to train a different model. Valid names: `model`, `model_1`, `model_2`.

In [ ]:
import pandas as pd
import torch

from data_prep import get_df_filtered
from dataloader import get_dataloaders
from engine import train
from helpers import create_model, save_csv, save_json
from models.crossflow import CharbonnierEndpointLoss
from plotting import plot_loss_curves

In [ ]:
MODEL_NAME = "model"

DATA_PATH = "/Users/kb/Documents/AI_in_mfg/AI_mfg_data"
DATA_FRAC = 0.3
TEST_SIZE = 0.2
BATCH_SIZE = 32
SEED = 42

LR = 1e-3
EPOCHS = 10
DEVICE = "mps"

In [ ]:
df_filtered = get_df_filtered(DATA_PATH)

train_dl, test_dl = get_dataloaders(
    df_filtered,
    data_frac=DATA_FRAC,
    test_size=TEST_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED,
)

In [ ]:
model = create_model(MODEL_NAME, pretrained=True)

model.name

In [ ]:
loss_func = CharbonnierEndpointLoss()
optimizer = torch.optim.Adam(
    lr=LR,
    params=model.parameters(),
)

In [ ]:
results = train(
    model=model,
    train_dataloader=train_dl,
    test_dataloader=test_dl,
    optimizer=optimizer,
    loss_fn=loss_func,
    epochs=EPOCHS,
    device=DEVICE,
    save_dir="models",
    model_name=f"{model.name}.pth",
)

In [ ]:
save_csv(pd.DataFrame(results), f"results/{model.name}.csv")
save_json(results, f"results/{model.name}.json")

In [ ]:
plot_loss_curves(results, save_path=f"results/{model.name}_loss_curves.png")